# ML-3: Random Forest Model Training & SMOTE Experimentation

This notebook trains and evaluates Random Forest failure prediction models on the full AI4I 2020 Predictive Maintenance Dataset (~10,000 rows).

## Key Design & Implementation Rules:
1. **Full Dataset Usage**: Uses the full dataset (`data/cleaned_predictive_maintenance.csv`). No small subsets.
2. **Stratified 80/20 Train/Test Split**: Performs one stratified split. The 20% test set is kept **completely untouched** for ML-4.
3. **Evaluated Approaches**:
   - **Approach A**: Balanced Random Forest (`class_weight='balanced'`)
   - **Approach B**: SMOTE + Random Forest (`SMOTE` inside `imblearn.pipeline`)
4. **Strict Leakage-Free SMOTE Rule**: SMOTE is applied **strictly inside each training fold** during cross-validation via `imblearn.pipeline.Pipeline`. SMOTE is **never** applied before splitting or to test/validation sets.
5. **5-Fold Stratified Cross-Validation**: Models are evaluated on the 80% training set across Recall, Precision, F1, PR-AUC, and ROC-AUC.
6. **Model Selection & Artifact Export**: The winning approach based on CV metrics (emphasizing PR-AUC and F1-Score) is fitted on the complete 80% training set and exported to `ml/models/best_model.joblib` for downstream ML-4 evaluation.

In [2]:
import os
import sys
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    roc_auc_score, average_precision_score, precision_score, 
    recall_score, f1_score
)

# Imbalanced-learn imports for leakage-free SMOTE inside CV folds
from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE

sys.path.append(os.path.abspath("../src"))
from feature_engineering import (
    DomainFeatureEngineer, 
    build_preprocessing_pipeline,
    EXCLUDED_COLS,
    TARGET_COL,
    BASE_FEATURE_COLS
)

## 1. Ingest Full Cleaned Dataset (~10,000 rows)

We load the full cleaned dataset and exclude leakage-prone failure-mode columns (`TWF`, `HDF`, `PWF`, `OSF`, `RNF`) and identifiers (`UDI`, `Product ID`).

In [3]:
data_path = "../../data/cleaned_predictive_maintenance.csv"
if not os.path.exists(data_path):
    data_path = "../../data/ai4i2020.csv"

df = pd.read_csv(data_path)
print(f"Loaded full dataset shape: {df.shape}")

feature_cols = [c for c in df.columns if c not in EXCLUDED_COLS and c != TARGET_COL]
X = df[feature_cols].copy()
y = df[TARGET_COL].copy()

print(f"Predictive Input Features ({len(feature_cols)}): {feature_cols}")
print(f"Target ({TARGET_COL}) Class Distribution:")
print(y.value_counts(normalize=True))

Loaded full dataset shape: (10000, 7)
Predictive Input Features (6): ['Type', 'Air temperature [K]', 'Process temperature [K]', 'Rotational speed [rpm]', 'Torque [Nm]', 'Tool wear [min]']
Target (Machine failure) Class Distribution:
Machine failure
0    0.9661
1    0.0339
Name: proportion, dtype: float64


## 2. Stratified 80/20 Train/Test Split (Untouched Test Set)

We perform a single stratified 80/20 train/test split. The 20% test set is preserved completely untouched for final evaluation in ML-4.

In [4]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

print(f"X_train shape: {X_train.shape} (Failures: {y_train.sum()})")
print(f"X_test shape:  {X_test.shape} (Failures: {y_test.sum()} - Untouched for ML-4)")

X_train shape: (8000, 6) (Failures: 271)
X_test shape:  (2000, 6) (Failures: 68 - Untouched for ML-4)


## 3. Define Candidate Pipelines & Leakage-Free SMOTE Design

We construct two candidate pipelines using `imblearn.pipeline.Pipeline`:
1. **Approach A (Balanced Random Forest)**: Uses `class_weight='balanced'` in Random Forest.
2. **Approach B (SMOTE + Random Forest)**: Embeds `SMOTE` inside the pipeline **after** domain engineering and preprocessing, ensuring SMOTE runs strictly on training folds.

In [5]:
# Approach A: Balanced Random Forest Baseline
pipeline_a = ImbPipeline([
    ('engineer', DomainFeatureEngineer()),
    ('preprocessor', build_preprocessing_pipeline()),
    ('classifier', RandomForestClassifier(
        n_estimators=100, class_weight='balanced', random_state=42, n_jobs=-1
    ))
])

# Approach B: SMOTE + Random Forest
pipeline_b = ImbPipeline([
    ('engineer', DomainFeatureEngineer()),
    ('preprocessor', build_preprocessing_pipeline()),
    ('smote', SMOTE(random_state=42)),
    ('classifier', RandomForestClassifier(
        n_estimators=100, random_state=42, n_jobs=-1
    ))
])

candidate_pipelines = {
    "Balanced Random Forest": pipeline_a,
    "SMOTE + Random Forest": pipeline_b
}

## 4. 5-Fold Stratified Cross-Validation on Training Set

We evaluate both candidate approaches using **5-Fold Stratified K-Fold CV** strictly on the training set.

In [6]:
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = {
    'recall': 'recall',
    'precision': 'precision',
    'f1': 'f1',
    'pr_auc': 'average_precision',
    'roc_auc': 'roc_auc'
}

cv_comparison = []

for name, pipeline in candidate_pipelines.items():
    cv_res = cross_validate(pipeline, X_train, y_train, cv=skf, scoring=scoring, n_jobs=-1)
    
    rec_mean = np.mean(cv_res['test_recall'])
    prec_mean = np.mean(cv_res['test_precision'])
    f1_mean = np.mean(cv_res['test_f1'])
    pr_auc_mean = np.mean(cv_res['test_pr_auc'])
    roc_auc_mean = np.mean(cv_res['test_roc_auc'])
    
    cv_comparison.append({
        "Model": name,
        "CV Recall": round(rec_mean, 4),
        "CV Precision": round(prec_mean, 4),
        "CV F1": round(f1_mean, 4),
        "CV PR-AUC": round(pr_auc_mean, 4),
        "CV ROC-AUC": round(roc_auc_mean, 4)
    })

comp_df = pd.DataFrame(cv_comparison)
print("Cross-Validation Comparison Table (Training Set Only):")
print(comp_df.to_string(index=False))

Cross-Validation Comparison Table (Training Set Only):
                 Model  CV Recall  CV Precision  CV F1  CV PR-AUC  CV ROC-AUC
Balanced Random Forest     0.8119        0.9238 0.8640     0.8872      0.9754
 SMOTE + Random Forest     0.8266        0.7197 0.7684     0.8592      0.9770


## 5. Model Selection & Complete Training Set Fitting

We select the winning pipeline based on CV metrics (emphasizing PR-AUC, F1, and Recall), fit it on the full 80% training set, and export the trained artifact to `ml/models/best_model.joblib`.

In [7]:
sorted_df = comp_df.sort_values(by=['CV PR-AUC', 'CV F1', 'CV Recall'], ascending=False)
winning_model_name = sorted_df.iloc[0]['Model']
winning_pipeline = candidate_pipelines[winning_model_name]

print(f"★ Selected Winner based on CV evidence: '{winning_model_name}' ★")

# Fit winning pipeline on the full training set (X_train, y_train)
winning_pipeline.fit(X_train, y_train)

# Extract recoverable feature names
preprocessor = winning_pipeline.named_steps['preprocessor']
feature_names = list(preprocessor.get_feature_names_out())

models_dir = "../models"
os.makedirs(models_dir, exist_ok=True)

model_artifact = {
    "pipeline": winning_pipeline,
    "model_name": winning_model_name,
    "feature_names": feature_names,
    "input_feature_cols": feature_cols,
    "cv_metrics": sorted_df.iloc[0].to_dict()
}

best_model_path = os.path.join(models_dir, "best_model.joblib")
joblib.dump(model_artifact, best_model_path)
print(f"Saved trained winning pipeline artifact to: {os.path.abspath(best_model_path)}")

cv_json_path = os.path.join(models_dir, "cv_model_comparison.json")
with open(cv_json_path, "w") as f:
    json.dump({
        "winning_model": winning_model_name,
        "cv_comparison": cv_comparison
    }, f, indent=2)
print(f"Saved CV comparison JSON to: {os.path.abspath(cv_json_path)}")

★ Selected Winner based on CV evidence: 'Balanced Random Forest' ★
Saved trained winning pipeline artifact to: c:\Users\bingu\OneDrive\Desktop\cognizant\predictive-maintenance\ml\models\best_model.joblib
Saved CV comparison JSON to: c:\Users\bingu\OneDrive\Desktop\cognizant\predictive-maintenance\ml\models\cv_model_comparison.json
